<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML/W13/Q1/ASSIGNMNT_W12_Q_1_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# IIT JODHPUR BSc - TASK 1
# MANUAL PRINCIPAL COMPONENT ANALYSIS (PCA) PROJECTION
#
# Deliverables:
# 1. pca_calculation.pdf
# 2. pca_verify.py
# 3. pca_output.png
# ============================================================

import os
import sys
import json
import subprocess
import numpy as np

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Preformatted
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
import matplotlib.pyplot as plt

# ============================================================
# STEP 1: INPUT DATA
# ============================================================

print("=" * 60)
print("PCA - PRINCIPAL COMPONENT PROJECTION")
print("=" * 60)

# Provided assignment constants
MU1 = 25.0
SIGMA1 = 11.18

print("\nProvided constants:")
print("Mean (Feature 1):", MU1)
print("Standard Deviation (Feature 1):", SIGMA1)

# Enter the remaining values from your assignment
print("\nEnter the remaining assignment values:")

X1 = float(input("Enter data point X1: "))
X2 = float(input("Enter data point X2: "))

MU2 = float(input("Enter Feature 2 mean (mu2): "))
SIGMA2 = float(input("Enter Feature 2 standard deviation (sigma2): "))

PC1_1 = float(input("Enter PC1 eigenvector component 1: "))
PC1_2 = float(input("Enter PC1 eigenvector component 2: "))

# ============================================================
# STEP 2: VALIDATE INPUTS
# ============================================================

if SIGMA1 <= 0 or SIGMA2 <= 0:
    raise ValueError("Standard deviations must be positive.")

if np.linalg.norm([PC1_1, PC1_2]) == 0:
    raise ValueError("PC1 eigenvector cannot be zero.")

# Normalize eigenvector to ensure unit length
pc1 = np.array([[PC1_1], [PC1_2]], dtype=float)
pc1 = pc1 / np.linalg.norm(pc1)

# ============================================================
# STEP 3: CREATE PYTHON VERIFICATION SCRIPT
# ============================================================

config = {
    "x1": X1,
    "x2": X2,
    "mu1": MU1,
    "mu2": MU2,
    "sigma1": SIGMA1,
    "sigma2": SIGMA2,
    "pc1_1": float(pc1[0, 0]),
    "pc1_2": float(pc1[1, 0])
}

script = r'''
import numpy as np

# Input parameters
DATA_POINT = np.array([[{x1}, {x2}]])

MEAN = np.array([[{mu1}, {mu2}]])
STD = np.array([[{sigma1}, {sigma2}]])

PC1 = np.array([
    [{pc1_1}],
    [{pc1_2}]
])

print("=" * 55)
print("PCA PROJECTION VERIFICATION")
print("=" * 55)

# Step 1: Standardization
STANDARDIZED = (DATA_POINT - MEAN) / STD

print("\n1. ORIGINAL DATA POINT (1x2):")
print(DATA_POINT)

print("\n2. MEAN MATRIX:")
print(MEAN)

print("\n3. STANDARD DEVIATION MATRIX:")
print(STD)

print("\n4. STANDARDIZED DATA POINT:")
print(STANDARDIZED)

# Step 2: Principal Component
print("\n5. FIRST PRINCIPAL COMPONENT (2x1):")
print(PC1)

# Step 3: Matrix multiplication
print("\n6. MATRIX MULTIPLICATION:")
print("(1x2) x (2x1) = (1x1)")

PROJECTED = np.matmul(STANDARDIZED, PC1)

print("\n7. FINAL 1D PROJECTED COORDINATE:")
print(PROJECTED)

print("\nFinal Answer:", round(float(PROJECTED[0, 0]), 6))

print("\n" + "=" * 55)
print("PCA PROJECTION COMPLETED SUCCESSFULLY")
print("=" * 55)
'''.format(**config)

with open("pca_verify.py", "w") as file:
    file.write(script)

print("\nPython verification script created.")

# ============================================================
# STEP 4: EXECUTE PYTHON SCRIPT
# ============================================================

result = subprocess.run(
    [sys.executable, "pca_verify.py"],
    capture_output=True,
    text=True,
    check=True
)

output = result.stdout

print("\n" + output)

# ============================================================
# STEP 5: GENERATE PDF REPORT
# ============================================================

# Calculate values for the report
data_point = np.array([[X1, X2]])
mean = np.array([[MU1, MU2]])
std = np.array([[SIGMA1, SIGMA2]])

standardized = (data_point - mean) / std

projection = np.matmul(standardized, pc1)

final_value = float(projection[0, 0])

pdf = SimpleDocTemplate(
    "pca_calculation.pdf",
    pagesize=A4,
    rightMargin=55,
    leftMargin=55,
    topMargin=50,
    bottomMargin=50
)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "TitleCustom",
    parent=styles["Title"],
    fontSize=17,
    leading=23,
    textColor=colors.darkblue,
    spaceAfter=20
)

heading_style = ParagraphStyle(
    "HeadingCustom",
    parent=styles["Heading2"],
    fontSize=12,
    leading=16,
    textColor=colors.darkblue,
    spaceBefore=15,
    spaceAfter=8
)

body_style = styles["BodyText"]

code_style = ParagraphStyle(
    "CodeCustom",
    fontName="Courier",
    fontSize=9,
    leading=14,
    leftIndent=10,
    spaceAfter=10
)

story = []

story.append(Paragraph(
    "Principal Component Analysis (PCA)",
    title_style
))

story.append(Paragraph(
    "<b>Task 1: Calculate Principal Component Projection</b>",
    body_style
))

story.append(Spacer(1, 15))

story.append(Paragraph(
    "1. Original Data and Statistical Parameters",
    heading_style
))

story.append(Preformatted(
    f"""
Data Point X = [{X1}, {X2}]

Mean:
mu = [{MU1}, {MU2}]

Standard Deviation:
sigma = [{SIGMA1}, {SIGMA2}]

First Principal Component:
PC1 = [{pc1[0,0]:.6f}, {pc1[1,0]:.6f}]^T
""",
    code_style
))

story.append(Paragraph(
    "2. Standardization of Data",
    heading_style
))

story.append(Paragraph(
    "Each feature is standardized using the formula:",
    body_style
))

story.append(Preformatted(
    "Z = (X - mu) / sigma",
    code_style
))

story.append(Paragraph(
    "Feature 1 calculation:",
    body_style
))

story.append(Preformatted(
    f"""
Z1 = ({X1} - {MU1}) / {SIGMA1}
Z1 = {standardized[0,0]:.8f}
""",
    code_style
))

story.append(Paragraph(
    "Feature 2 calculation:",
    body_style
))

story.append(Preformatted(
    f"""
Z2 = ({X2} - {MU2}) / {SIGMA2}
Z2 = {standardized[0,1]:.8f}
""",
    code_style
))

story.append(Paragraph(
    "Standardized Data Matrix:",
    body_style
))

story.append(Preformatted(
    f"""
Z = [{standardized[0,0]:.8f}, {standardized[0,1]:.8f}]
""",
    code_style
))

story.append(Paragraph(
    "3. Principal Component Projection",
    heading_style
))

story.append(Paragraph(
    "The standardized 1x2 row vector is multiplied "
    "by the 2x1 first principal component column vector.",
    body_style
))

story.append(Preformatted(
    f"""
Z (1x2) =
[{standardized[0,0]:.8f}, {standardized[0,1]:.8f}]

PC1 (2x1) =
[{pc1[0,0]:.8f}]
[{pc1[1,0]:.8f}]

Projection = Z x PC1

Projection =
({standardized[0,0]:.8f} x {pc1[0,0]:.8f})
+
({standardized[0,1]:.8f} x {pc1[1,0]:.8f})

Projection = {final_value:.8f}
""",
    code_style
))

story.append(Paragraph(
    "4. Final Result",
    heading_style
))

story.append(Paragraph(
    f"<b>Final 1D Projected Coordinate = {final_value:.8f}</b>",
    body_style
))

story.append(Spacer(1, 15))

story.append(Paragraph(
    "Conclusion: The original two-dimensional data point "
    "has been standardized and projected onto the first "
    "principal component, resulting in a one-dimensional "
    "representation.",
    body_style
))

pdf.build(story)

print("\nPDF report generated: pca_calculation.pdf")

# ============================================================
# STEP 6: GENERATE OUTPUT SCREENSHOT
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

ax.axis("off")

ax.text(
    0.02, 0.97,
    "PCA PROJECTION - SCRIPT OUTPUT",
    fontsize=17,
    fontweight="bold",
    va="top"
)

ax.text(
    0.02, 0.90,
    output,
    fontsize=9,
    family="monospace",
    va="top"
)

plt.savefig(
    "pca_output.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close()

print("Output screenshot generated: pca_output.png")

# ============================================================
# STEP 7: DISPLAY GENERATED FILES
# ============================================================

print("\n" + "=" * 60)
print("ALL DELIVERABLES GENERATED")
print("=" * 60)

files = [
    "pca_calculation.pdf",
    "pca_verify.py",
    "pca_output.png"
]

for filename in files:
    print("[SUCCESS]", filename, "-", os.path.getsize(filename), "bytes")

print("\nFinal PCA Coordinate:", round(final_value, 8))

# ============================================================
# STEP 8: DOWNLOAD ALL DELIVERABLES
# ============================================================

from google.colab import files

for filename in files:
    files.download(filename)